To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your own computer, follow the installation instructions on our Github page [here](https://docs.unsloth.ai/get-started/installing-+-updating).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & [how to save it](#Save)


### News

**Read our [blog post](https://unsloth.ai/blog/r1-reasoning) for guidance on how to train reasoning models.**

Visit our docs for all our [model uploads](https://docs.unsloth.ai/get-started/all-our-models) and [notebooks](https://docs.unsloth.ai/get-started/unsloth-notebooks).


### Installation

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "5"

In [3]:
%%capture
!apt-get install swi-prolog

In [1]:
!pip install -U pyswip

  Using cached pyswip-0.3.2-py3-none-any.whl (35 kB)


In [2]:
%%capture
# Skip restarting message in Colab
import sys; modules = list(sys.modules.keys())
for x in modules: sys.modules.pop(x) if "PIL" in x or "google" in x else None



In [ ]:
!pip install unsloth vllm
!pip install --upgrade pillow

### Unsloth

Use `PatchFastRL` before all functions to patch GRPO and other RL algorithms!

In [3]:
from unsloth import FastLanguageModel, PatchFastRL
PatchFastRL("GRPO", FastLanguageModel)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/lab/biancaraimondi/LLM_Format/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!


2025-02-24 14:18:41,604	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


Load up `Llama 3.1 8B Instruct`, and set parameters

In [4]:
from unsloth import is_bfloat16_supported
import torch
max_seq_length = 2048 # Can increase for longer reasoning traces
lora_rank = 32 # Larger rank = smarter, but slower

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-Coder-3B-Instruct",
    max_seq_length = max_seq_length,
    load_in_4bit = True, # False for LoRA 16bit
    fast_inference = True, # Enable vLLM fast inference
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.6, # Reduce if out of memory
)

model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ], # Remove QKVO if out of memory
    lora_alpha = lora_rank,
    use_gradient_checkpointing = "unsloth", # Enable long context finetuning
    random_state = 3407,
)

INFO 02-24 14:18:44 __init__.py:207] Automatically detected platform cuda.
==((====))==  Unsloth 2025.2.15: Fast Qwen2 patching. Transformers: 4.49.0.
   \\   /|    GPU: NVIDIA A100-SXM4-80GB. Max memory: 79.151 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.28.post3. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading unsloth/qwen2.5-coder-3b-instruct-bnb-4bit with actual GPU utilization = 59.69%
Unsloth: Your GPU has CUDA compute capability 8.0 with VRAM = 79.15 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 2048. Num Sequences = 226.
Unsloth: vLLM's KV Cache can use up to 44.94 GB. Also swap space = 6 GB.
INFO 02-24 14:18:58 config.py:549] This model supports multiple tasks: {'reward', 'generate', 'embed', 'classify', 'scor

[W224 14:19:00.884975246 CUDAAllocatorConfig.h:28] Warning: expandable_segments not supported on this platform (function operator())


INFO 02-24 14:19:01 weight_utils.py:254] Using model weights format ['*.safetensors']


Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:03<00:00,  3.32s/it]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:03<00:00,  3.32s/it]

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  1.97it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  1.96it/s]



INFO 02-24 14:19:05 model_runner.py:1115] Loading model weights took 1.9356 GB
INFO 02-24 14:19:05 punica_selector.py:18] Using PunicaWrapperGPU.
INFO 02-24 14:19:16 worker.py:267] Memory profiling takes 8.23 seconds
INFO 02-24 14:19:16 worker.py:267] the current vLLM instance can use total_gpu_memory (79.15GiB) x gpu_memory_utilization (0.60) = 47.24GiB
INFO 02-24 14:19:16 worker.py:267] model weights take 1.94GiB; non_torch_memory takes 0.09GiB; PyTorch activation peak memory takes 1.24GiB; the rest of the memory reserved for KV Cache is 43.97GiB.
INFO 02-24 14:19:17 executor_base.py:111] # cuda blocks: 80051, # CPU blocks: 10922
INFO 02-24 14:19:17 executor_base.py:116] Maximum concurrency for 2048 tokens per request: 625.40x
INFO 02-24 14:19:28 model_runner.py:1434] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI. If out-of-memory erro

Capturing CUDA graph shapes: 100%|██████████| 32/32 [01:04<00:00,  2.00s/it]

INFO 02-24 14:20:32 model_runner.py:1562] Graph capturing finished in 64 secs, took 4.70 GiB
INFO 02-24 14:20:32 llm_engine.py:436] init engine (profile, create kv cache, warmup model) took 84.40 seconds



Unsloth 2025.2.15 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


### Data Prep
<a name="Data"></a>

We directly leverage [@willccbb](https://gist.github.com/willccbb/4676755236bb08cab5f4e54a0475d6fb) for data prep and all reward functions. You are free to create your own!

In [5]:
import re
from datasets import load_dataset, Dataset
import ast
import concurrent
import os
from pyswip import Prolog

os.environ["WANDB_PROJECT"] = "python-3b"
os.environ["WANDB_LOG_MODEL"] = "checkpoint"

# Load and prep dataset
SYSTEM_PROMPT = """
Generate a prolog solution for the asked question.
Follow these steps to craft your response:
1. reason about the given instruction
2. provide a high-quality prolog solution
3. write a query to verify the solution.
Output in the following format:
<reasoning>
...
</reasoning>
<knowledge>
...
</knowledge>
<query>
...
</query>

Write the test case just inside <query></query> not in <code></code>.
"""

def extract_xml_knowledge(text: str) -> str:
    answer = text.split("<knowledge>")[-1]
    answer = answer.split("</knowledge>")[0]
    return answer.strip()

def extract_xml_query(text: str) -> str:
    answer = text.split("<query>")[-1]
    answer = answer.split("</query>")[0]
    return answer.strip()

def extract_hash_answer(text: str) -> str | None:
    if "####" not in text:
        return None
    return text.split("####")[1].strip()

# uncomment middle messages for 1-shot prompting
def get_gsm8k_questions(split = "train") -> Dataset:
    data = load_dataset('openai/gsm8k', 'main')[split] # type: ignore
    data = data.map(lambda x: { # type: ignore
        'prompt': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': x['question']}
        ],
        'answer': extract_hash_answer(x['answer'])
    }) # type: ignore
    return data # type: ignore

dataset = get_gsm8k_questions()

def parse_kb(prolog_code, query, answer):
  try:
    prolog_interpreter = Prolog()
    #for rule in prolog_code.split("."):
    #  prolog_interpreter.assertz(rule)
    prolog_interpreter.consult("knowledge_base.pl")
    value = prolog_interpreter.query(query)
    print(value)
    return value == int(answer)
  except:
    return 0

import re
 
def remove_prolog_comments_and_whitespace(code):
    # Remove block comments (/* ... */)
    code_no_block = re.sub(r'/\*[\s\S]*?\*/', '', code)
    # Remove single-line comments (% ...) from each line
    code_no_comments = re.sub(r'(?m)%.*$', '', code_no_block)
    # Remove all whitespace characters (spaces, newlines, tabs)
    cleaned_code = re.sub(r'\s+', '', code_no_comments)
    return cleaned_code

# Reward functions
def correctness_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    responses = [completion[0]['content'] for completion in completions]
    q = prompts[0][-1]['content']
    reward = []
    for r in responses:
      print(r)
      knowledge_base = extract_xml_knowledge(r)
      knowledge_base = knowledge_base.replace("```prolog", "")
      knowledge_base = knowledge_base.replace("```", "")
      query = extract_xml_query(r)
      query = query.replace("```prolog", "")
      query = query.replace("```", "")
      with open("knowledge_base.pl", "w") as f:
        f.write(knowledge_base)
      reward_achieved = parse_kb(knowledge_base, remove_prolog_comments_and_whitespace(query), answer)
      # check with ast if code can be parsed
      reward.append(reward_achieved)
    print('-'*20, f"Question:\n{q}", f"\nAnswer:\n{answer[-1]}", f"\nResponse:\n{responses[-1]}")
    return reward  #[2.0 if r == a else 0.0 for r, a in zip(extracted_responses, answer)]

def count_xml(text) -> float:
    count = 0.0
    if text.count("<reasoning>\n") == 1:
        count += 0.125
    if text.count("\n</reasoning>\n") == 1:
        count += 0.125
    if text.count("<knowledge>\n") == 1:
        count += 0.125
    if text.count("\n</knowledge>\n") == 1:
        count += 0.125
    if text.count("\n<query>\n") == 1:
        count += 0.125
        count -= len(text.split("\n</query>\n")[-1])*0.001
    if text.count("\n</query>") == 1:
        count += 0.125
        count -= (len(text.split("\n</query>")[-1]) - 1)*0.001
    return count

def xmlcount_reward_func(completions, **kwargs) -> list[float]:
    contents = [completion[0]["content"] for completion in completions]
    return [count_xml(c) for c in contents]

<a name="Train"></a>
### Train the model

Now set up GRPO Trainer and all configurations!

In [6]:
from trl import GRPOConfig, GRPOTrainer
training_args = GRPOConfig(
    use_vllm = True, # use vLLM for fast inference!
    learning_rate = 5e-6,
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    weight_decay = 0.1,
    warmup_ratio = 0.1,
    lr_scheduler_type = "cosine",
    optim = "paged_adamw_8bit",
    logging_steps = 1,
    bf16 = is_bfloat16_supported(),
    fp16 = not is_bfloat16_supported(),
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 1, # Increase to 4 for smoother training
    num_generations = 4, # Decrease if out of memory
    max_prompt_length = 256,
    max_completion_length = 1024,
    # num_train_epochs = 1, # Set to 1 for a full training run
    max_steps = 250,
    save_steps = 250,
    max_grad_norm = 0.1,
    report_to = None, #"wandb", # Can use Weights & Biases
    output_dir = "outputs",
)

Unsloth: We now expect `per_device_train_batch_size` to be a multiple of `num_generations`.
We will change the batch size of 1 to the `num_generations` of 4


And let's run the trainer! If you scroll up, you'll see a table of rewards. The goal is to see the `reward` column increase!

You might have to wait 150 to 200 steps for any action. You'll probably get 0 reward for the first 100 steps. Please be patient!

| Step | Training Loss | reward    | reward_std | completion_length | kl       |
|------|---------------|-----------|------------|-------------------|----------|
| 1    | 0.000000      | 0.125000  | 0.000000   | 200.000000        | 0.000000 |
| 2    | 0.000000      | 0.072375  | 0.248112   | 200.000000        | 0.000000 |
| 3    | 0.000000      | -0.079000 | 0.163776   | 182.500000        | 0.000005 |


In [ ]:
trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        xmlcount_reward_func,
        correctness_reward_func,
    ],
    args = training_args,
    train_dataset = dataset,
)
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs = 1
   \\   /|    Num examples = 7,473 | Num Epochs = 1
O^O/ \_/ \    Batch size per device = 4 | Gradient Accumulation steps = 1
\        /    Total batch size = 4 | Total steps = 250
 "-____-"     Number of trainable parameters = 59,867,136


<reasoning>
To determine how much Mr. Benson paid, we need to account for the special discount he received for buying more than 10 tickets. Here's a step-by-step breakdown:

1. Calculate the cost of 12 tickets without any discount.
2. Identify how many tickets exceed 10 and apply the 5% discount to those tickets.
3. Calculates the total cost after applying the discount.

Let's translate these steps into Prolog.
</reasoning>

<knowledge>
:- module(ticket_discount, [calculate_total_cost/1]).

% cost per ticket
COST_PER_TICKET is 40.

% initial number of tickets
TICKETS is 12.

% discount percentage
DISCOUNT_PERCENT is 0.05.

calculate_total_cost(TotalCost) :-
    % Calculate the number of tickets above 10
    TicketsOverTen is max(0, TICKETS - 10),
    
    % Calculate the cost before discount
    TotalCostBeforeDiscount is COST_PER_TICKET * TICKETS,
    
    % Calculate the discount amount for tickets over 10
    DiscountAmount is TicketsOverTen * COST_PER_TICKET * DISCOUNT_PERCENT,
   

ERROR: /home/lab/biancaraimondi/LLM_Format/knowledge_base.pl:4:
ERROR:    No permission to modify static procedure `(is)/2'
ERROR: /home/lab/biancaraimondi/LLM_Format/knowledge_base.pl:7:
ERROR:    No permission to modify static procedure `(is)/2'
ERROR: /home/lab/biancaraimondi/LLM_Format/knowledge_base.pl:10:
ERROR:    No permission to modify static procedure `(is)/2'
ERROR: /home/lab/biancaraimondi/LLM_Format/knowledge_base.pl:2:3: Syntax error: Operator expected
ERROR: /home/lab/biancaraimondi/LLM_Format/knowledge_base.pl:2:33: Syntax error: Operator expected
ERROR: /home/lab/biancaraimondi/LLM_Format/knowledge_base.pl:3:7: Syntax error: Operator expected
ERROR: /home/lab/biancaraimondi/LLM_Format/knowledge_base.pl:4:
ERROR:    Type error: `callable' expected, found `2' (an integer)
ERROR: /home/lab/biancaraimondi/LLM_Format/knowledge_base.pl:4:
ERROR:    Arguments are not sufficiently instantiated


<generator object Prolog._QueryWrapper.__call__ at 0x70af0396b680>
<reasoning>
To solve this problem, we need to calculate the total cost of the tickets after applying the discount. Here's a step-by-step breakdown of how we can approach this problem in Prolog:

1. **Define the cost of a single ticket**: $40.
2. **Determine the discount rate**: 5%.
3. **Calculate the discount for tickets bought beyond 10**: This means the first 10 tickets are not discounted, but any tickets bought after 10 are discounted at 5%.
4. **Calculate the total cost**: Sum the costs of the first 10 tickets and the discounted tickets.
5. **Output the total amount paid**.

Now, let's implement this in Prolog.
</reasoning>
<knowledge>
```prolog
% Define the cost of a single ticket
ticket_cost(40).

% Define the discount rate
discount_rate(0.05).

% Calculate the total cost of tickets after applying the discount
total_cost(NUMBER_OF_TICKETS) :-
    ticket_cost(TICKET_COST),
    DISCOUNTED_TICKETS is max(NUMBER_OF_TI

ERROR: /home/lab/biancaraimondi/LLM_Format/knowledge_base.pl:4:
ERROR:    Arguments are not sufficiently instantiated


-------------------- Question:
A concert ticket costs $40. Mr. Benson bought 12 tickets and received a 5% discount for every ticket bought that exceeds 10. How much did Mr. Benson pay in all? 
Answer:
476 
Response:
### Reasoning
To determine how much Mr. Benson paid for the concert tickets, we need to consider the following:
1. The cost of one ticket is $40.
2. Mr. Benson bought 12 tickets.
3. If the number of tickets exceeds 10, there is a 5% discount on each ticket.

First, we calculate the total cost without any discount. Then, we apply the discount to the tickets that exceed 10.

### Knowledge
- `price_of_ticket` is 40
- `num_tickets` is 12
- `discount_rate` is 5%

### Solution
```prolog
cost_of_ticket(40).
num_tickets(12).
discount_rate(5).

total_price :-
    total_cost_without_discount(WithoutDiscount),
    apply_discount(WithoutDiscount, Discounted),
    write(Discounted).

total_cost_without_discount(WithoutDiscount) :-
    price_of_ticket(Price),
    num_tickets(Tickets),
  

Step,Training Loss,reward,reward_std,completion_length,kl,rewards / xmlcount_reward_func,rewards / correctness_reward_func


<reasoning>
In this problem, we need to determine the monthly payment for both the house and the trailer, considering a 20-year loan, and then find the difference in these payments. We'll use the formula for monthly mortgage payments, which is:

\[ M = P \frac{r(1 + r)^n}{(1 + r)^n - 1} \]

Where:
- \( M \) is the monthly payment,
- \( P \) is the principal (loan amount),
- \( r \) is the monthly interest rate,
- \( n \) is the number of payments (loan term in months).

We know:
- The loan term is 20 years, which is \( 20 \times 12 = 240 \) months.
- The interest rate is typically 5% per year, or \( 0.05/12 \) per month.

We'll write a Prolog program to calculate the monthly payment for both the house and the trailer and then find the difference.
</reasoning>
<knowledge>
loan_amount/3 floats  - calculates the monthly payment using the formula provided.
200.00/3 - common monthly interest rate
</knowledge>
<query>
?- loan_amount(480000, 0.05/12, 240).
?- loan_amount(120000, 0.05/12, 240)

   Call: (1) pyrun("consult('knowledge_base.pl')", _2824) ? 

<a name="Inference"></a>
### Inference
Now let's try the model we just trained! First, let's first try the model without any GRPO trained:

In [ ]:
text = tokenizer.apply_chat_template([
    {"role" : "user", "content" : "Calculate pi."},
], tokenize = False, add_generation_prompt = True)

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 0.8,
    top_p = 0.95,
    max_tokens = 1024,
)
output = model.fast_generate(
    [text],
    sampling_params = sampling_params,
    lora_request = None,
)[0].outputs[0].text

output

Processed prompts: 100%|██████████| 1/1 [00:23<00:00, 23.78s/it, est. speed input: 1.64 toks/s, output: 19.94 toks/s]


'Calculating pi to a large number of decimal places is a complex task that requires a computational approach, rather than a simple mathematical formula. Here\'s a way to calculate pi using the Monte Carlo method, which is an approximation method that uses random numbers to estimate the value of pi:\n\n**The Monte Carlo Method**\n\nThe Monte Carlo method is based on the idea of simulating the probability of a random walk across a square and circle. Here\'s the basic idea:\n\n1. Draw a square and a circle on a piece of paper.\n2. Generate random points within the square.\n3. Count the proportion of points that fall within the circle.\n4. The ratio of points within the circle to the total number of points is approximately equal to the ratio of the area of the circle to the area of the square, which is pi.\n\n**Mathematical Formulation**\n\nLet\'s denote the following variables:\n\n*   `N`: the number of random points generated\n*   `n`: the number of points within the circle\n*   `pi_appr

And now with the LoRA we just trained with GRPO - we first save the LoRA first!

In [ ]:
model.save_lora("grpo_saved_lora")

Now we load the LoRA and test:

In [ ]:
text = tokenizer.apply_chat_template([
    {"role" : "system", "content" : SYSTEM_PROMPT},
    {"role" : "user", "content" : "Calculate pi."},
], tokenize = False, add_generation_prompt = True)

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 0.8,
    top_p = 0.95,
    max_tokens = 1024,
)
output = model.fast_generate(
    text,
    sampling_params = sampling_params,
    lora_request = model.load_lora("grpo_saved_lora"),
)[0].outputs[0].text

output

Processed prompts: 100%|██████████| 1/1 [00:23<00:00, 23.29s/it, est. speed input: 2.62 toks/s, output: 19.41 toks/s]


"<reasoning>\nPi (π) is an irrational number that represents the ratio of a circle's circumference to its diameter. It is approximately equal to 3.14159, but its decimal representation goes on indefinitely without repeating.\n\nTo calculate pi, we can use various mathematical formulas and methods, such as the Leibniz formula, the Gregory-Leibniz series, or the Monte Carlo method. However, these methods are not practical for obtaining a high degree of accuracy.\n\nA more practical approach is to use the Bailey-Borwein-Plouffe (BBP) formula, which is a spigot algorithm that allows us to calculate any digit of pi without having to compute the preceding digits.\n\nAnother method is to use the Chudnovsky algorithm, which is a fast and efficient method for calculating pi to a high degree of accuracy.\n\nFor simplicity, we can use the first few terms of the BBP formula to estimate pi:\nπ = 3 + 1/(4/3 - 1/(4/3 - 1/(4/3 - ...))\n\nLet's use this simplified formula to estimate pi:\n\nπ ≈ 3 + 1/(

Our reasoning model is much better - it's not always correct, since we only trained it for an hour or so - it'll be better if we extend the sequence length and train for longer!

<a name="Save"></a>
### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens.

In [ ]:
# Merge to 16bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_16bit", token = "")

# Merge to 4bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_4bit", token = "")

# Just LoRA adapters
if False: model.save_pretrained_merged("model", tokenizer, save_method = "lora",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "lora", token = "")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [Wiki page](https://github.com/unslothai/unsloth/wiki#gguf-quantization-options)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/drive/1WZDi7APtQ9VsvOrQSSC5DDtxq159j8iZ?usp=sharing)

In [ ]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("model", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("hf/model", tokenizer, token = "")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "f16", token = "")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "q4_k_m", token = "")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "hf/model", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "",
    )

Now, use the `model-unsloth.gguf` file or `model-unsloth-Q4_K_M.gguf` file in llama.cpp or a UI based system like Jan or Open WebUI. You can install Jan [here](https://github.com/janhq/jan) and Open WebUI [here](https://github.com/open-webui/open-webui)

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other links:
1. Llama 3.2 Conversational notebook. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(1B_and_3B)-Conversational.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
6. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://docs.unsloth.ai/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>
